In [ ]:
#Task 1
import pandas as pd

# Load dataset
df = pd.read_csv('/mnt/data/q3_retail_promotions.csv')

# Convert transaction_date to datetime
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

# Extract date features
df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.dayofweek  # Monday=0, Sunday=6

# Create is_month_end feature
df['is_month_end'] = df['transaction_date'].dt.day.apply(lambda x: 1 if x >= 25 else 0)

# Display sample
print(df.head())

In [ ]:
# Task 2
# Sort data by transaction_date
df = df.sort_values(by='transaction_date')

# Define split index (80% train, 20% test)
split_index = int(len(df) * 0.8)

# Split data (no randomization)
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

# Verify sizes
print("Training set size:", train_df.shape)
print("Test set size:", test_df.shape)

# Optional: Check date ranges
print("\nTrain date range:", train_df['transaction_date'].min(), "to", train_df['transaction_date'].max())
print("Test date range:", test_df['transaction_date'].min(), "to", test_df['transaction_date'].max())

In [ ]:
#Task 2 justification:
# The dataset is time-series in nature, with transactions occurring over time.
# Randomly shuffling the data would break the temporal order and lead to data leakage, making the model evaluation unrealistic.
# By splitting the data chronologically, we ensure that the model is trained on past data and tested on future data, which is a more accurate reflection of real-world performance.
# This approach allows us to evaluate how well the model can generalize to unseen data that follows the same temporal patterns as the training data.
# Additionally, using a time-based split helps to capture any seasonality or trends in the data, which can be crucial for modeling retail promotions effectively.

In [ ]:
#Task 3
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# -----------------------------
# Feature groups
# -----------------------------
categorical_features = ['promotion_type', 'location_type', 'store_size']

numerical_features = [
    'store_id', 'is_weekend', 'is_festival', 'competition_density',
    'year', 'month', 'day_of_week', 'is_month_end'
]

# -----------------------------
# Column Transformer
# -----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ]
)

# -----------------------------
# Pipeline
# -----------------------------
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

# -----------------------------
# Split features & target
# -----------------------------
X_train = train_df.drop(columns=['items_sold', 'transaction_date'])
y_train = train_df['items_sold']

X_test = test_df.drop(columns=['items_sold', 'transaction_date'])
y_test = test_df['items_sold']

# -----------------------------
# Fit ONLY on training data
# -----------------------------
pipeline.fit(X_train)

# -----------------------------
# Apply transformation
# -----------------------------
X_train_transformed = pipeline.transform(X_train)
X_test_transformed = pipeline.transform(X_test)

# -----------------------------
# Check output
# -----------------------------
print("Train shape:", X_train_transformed.shape)
print("Test shape:", X_test_transformed.shape)


In [ ]:
#Task 4
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -----------------------------
# Pipelines with models
# -----------------------------
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# -----------------------------
# Fit models
# -----------------------------
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

# -----------------------------
# Predictions
# -----------------------------
y_pred_lr = lr_pipeline.predict(X_test)
y_pred_rf = rf_pipeline.predict(X_test)

# -----------------------------
# Evaluation metrics
# -----------------------------
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print("Linear Regression → RMSE:", rmse_lr, "| MAE:", mae_lr)
print("Random Forest → RMSE:", rmse_rf, "| MAE:", mae_rf)

# -----------------------------
# Parity Plots
# -----------------------------
def parity_plot(y_true, y_pred, title):
    plt.figure()
    plt.scatter(y_true, y_pred)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val])  # diagonal line
    plt.xlabel("Actual items_sold")
    plt.ylabel("Predicted items_sold")
    plt.title(title)
    plt.show()

parity_plot(y_test, y_pred_lr, "Linear Regression Parity Plot")
parity_plot(y_test, y_pred_rf, "Random Forest Parity Plot")

# -----------------------------
# Feature Importances (Random Forest)
# -----------------------------
rf_model = rf_pipeline.named_steps['model']
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()

importances = rf_model.feature_importances_

# Create dataframe
feat_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

# Top 5 features
top5 = feat_importance_df.head(5)

print("\nTop 5 Important Features:")
print(top5)

In [ ]:
#Task 4 justification:
# Evaluating both a simple linear model and a more complex ensemble model allows us to compare their performance and understand the underlying patterns in the data.
# Linear Regression provides a baseline and is interpretable, while Random Forest can capture non-linear relationships and interactions between features.
#Features like competition_density, promotion_type, store_size, is_weekend, month often dominate
#These features strongly influence retail sales behavior,making them critical for accurate prdictions.